# 🧪 atomipy Visual Builder - Google Colab GPU Launch Guide
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mholmboe/atomipy-web-module/blob/main/ColabLaunchGuide.ipynb)

Welcome to the official launch manual for running the **atomipy Visual Builder** on **Google Colab** with full **GPU hardware acceleration**!

This notebook launches the visual interface in Colab's cloud environment so you can build complex mineral-water systems and run **OpenMM molecular dynamics simulations** on a high-performance **NVIDIA T4, L4, or A100 GPU**.

Unlike the public site at [www.atomipy.io](https://www.atomipy.io) (which is build-only), simulations are **enabled** here.

---

## ⚡ STEP 0: Enable GPU Acceleration

1. In the top-right menu of Colab, click **Runtime** -> **Change runtime type**.
2. Select **T4 GPU** (or a higher tier if available) under *Hardware accelerator*.
3. Click **Save**.

## 📦 STEP 1: Clone and Build the Application

This wipes any previous install, clones the repo, installs the Python dependencies (FastAPI + OpenMM + atomipy deps), and builds the React/Vite production frontend bundle.

In [ ]:
# 1. Reset directory and wipe previous folders
%cd /content
!rm -rf atomipy-web-module

# 2. Clone fresh
!git clone https://github.com/mholmboe/atomipy-web-module.git
%cd atomipy-web-module

# 3. Install Python dependencies (FastAPI backend + OpenMM + atomipy deps)
!pip install -q -r requirements.txt

# 4. Install Node and build the React frontend (served by FastAPI)
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
!apt-get install -y nodejs
!npm install --legacy-peer-deps
!npm run build

# 5. Localtunnel for the public URL
!npm install -g localtunnel

## 🚀 STEP 2: Launch the Tunnel and Visual Builder

This cell:
1. Retrieves your Colab instance's **external IP address** (your **Tunnel Password**).
2. Opens a secure Localtunnel on port `5002`.
3. Boots the **FastAPI** server, which serves both the API *and* the built frontend, with simulations **enabled** (GPU).

### Instructions
- 🔑 **Copy the IP address** printed below.
- 🔗 **Click the generated `Localtunnel` link**.
- 🔓 Paste the IP into the **"Tunnel Password"** field in your browser and submit.

> Note: the **Organic Molecule (GAFF/OpenFF)** node needs the separate OpenFF worker and is only available on the hosted site or a full local install — all other nodes and GPU simulations work here.

In [ ]:
import os
import subprocess
import time
import urllib.request

REPO = "/content/atomipy-web-module"
PORT = "5002"

# 1. Get this instance's IP (used as the Localtunnel password)
colab_ip = urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode("utf8").strip()

# 2. Start localtunnel in the background
lt_proc = subprocess.Popen(["lt", "--port", PORT], stdout=subprocess.PIPE, text=True)
time.sleep(3)
public_url = "Generating URL..."
for line in lt_proc.stdout:
    if "your url is:" in line:
        public_url = line.split("your url is:")[-1].strip()
        break

print("\n==================================================")
print(f"🔑 STEP 1 - COPY THIS PASSWORD: {colab_ip}")
print(f"👉 STEP 2 - CLICK THIS LINK: {public_url}")
print("==================================================\n")

# 3. Launch the FastAPI server (serves frontend + API; simulations enabled).
#    DISABLE_SIMULATION is left unset so MD/EM runs on the Colab GPU.
env = os.environ.copy()
env["PYTHONPATH"] = f"{REPO}:{REPO}/backend/core"
env["FRONTEND_DIST"] = f"{REPO}/dist"
subprocess.run(
    [
        "uvicorn", "main:app",
        "--app-dir", f"{REPO}/backend/core",
        "--host", "0.0.0.0",
        "--port", PORT,
    ],
    env=env,
)